# Démonstration du Modèle LMTL (Zooplancton)

Ce notebook démontre l'utilisation du nouveau module `seapopym.lmtl` pour simuler la dynamique d'une population de zooplancton (biomasse et production structurée par âge).

In [ ]:
from datetime import timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pint
import xarray as xr
from IPython.display import Markdown

from seapopym.blueprint import Blueprint
from seapopym.controller import SimulationConfig, SimulationController
from seapopym.lmtl.configuration import LMTLParams
from seapopym.lmtl.core import (
    compute_day_length,
    compute_gillooly_temperature,
    compute_layer_weighted_mean,
    compute_mortality_tendency,
    compute_production_dynamics,
    compute_production_initialization,
    compute_recruitment_age,
    compute_threshold_temperature,
)
from seapopym.standard.coordinates import Coordinates

In [ ]:
ureg = pint.get_application_registry()

## 1. Configuration

**Points clés:**
- Simulation sur 2 mois avec un pas de temps de 1 heure
- Température constante à 20°C (simplification)
- Production primaire constante à 1.0 g/m²/jour
- Modèle LMTL avec production structurée par âge (cohortes)

In [ ]:
# Configuration de la simulation
config = SimulationConfig(
    start_date="2020-01-01",
    end_date="2020-03-01",
    timestep=timedelta(hours=1),
)


# Paramètres avec unités explicites
lmtl_params = LMTLParams(
    day_layer=1,
    night_layer=1,
    tau_r_0=10.38 * ureg.days,  # Sera converti en secondes
    gamma_tau_r=ureg.Quantity(0.00, ureg.degC**-1),
    lambda_0=ureg.Quantity(1 / 10, ureg.day**-1),  # Sera converti en 1/s
    gamma_lambda=ureg.Quantity(0, ureg.degC**-1),
    E=1,  # Sans dimension
    T_ref=ureg.Quantity(0, ureg.degC),
)

## 2. Création de l'Environnement (Forçages)

**Important:** 
- Les forçages incluent: température, production primaire, cohortes, et dt
- Toutes les unités sont en SI (secondes, pas jours)
- dt et cohort sont des "forçages statiques" (ne varient pas dans le temps)

In [ ]:
times = np.arange(
    config.start_date,
    pd.to_datetime(config.end_date) + config.timestep,
    config.timestep,
    dtype="datetime64[ns]",
)
lats = np.array([0.0])  # Equateur
lons = np.array([0.0])
depths = np.array([10.0, 100.0])
cohorts = (np.arange(0, np.ceil(lmtl_params.tau_r_0.magnitude) + 1) * ureg.day).to("second")

print(f"Configuration temporelle:")
print(f"  Période: {times[0]} à {times[-1]}")
print(f"  Nombre de pas de temps: {len(times)}")
print(f"  Nombre de cohortes: {len(cohorts)}")
cohorts

In [ ]:
# Température : 20°C partout pour simplifier
temp_data = np.ones((len(times), len(lats), len(lons), len(depths))) * 20.0

temperature = xr.DataArray(
    temp_data,
    coords={
        Coordinates.T.value: times,
        Coordinates.Y.value: lats,
        Coordinates.X.value: lons,
        Coordinates.Z.value: depths,
    },
    dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value, Coordinates.Z.value),
    name="temperature",
    attrs={"units": "degree_Celsius"},
)

# Production Primaire (NPP) : Constante = 1.0 g/m²/day = 1.0/86400 g/m²/s
npp_value = 1.0 / 86400  # Conversion jour -> seconde
npp = xr.DataArray(
    np.ones((len(times), len(lats), len(lons))) * npp_value,
    coords={Coordinates.T.value: times, Coordinates.Y.value: lats, Coordinates.X.value: lons},
    dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
    name="primary_production",
    attrs={"units": "g/m**2/second"},
)

# Cohortes et dt comme forçages
cohorts_da = xr.DataArray(
    cohorts.magnitude, dims=["cohort"], name="cohort", attrs={"units": "second"}
)

dt_da = xr.DataArray(config.timestep.total_seconds(), name="dt", attrs={"units": "second"})

forcings = xr.Dataset(
    {
        "temperature": temperature,
        "primary_production": npp,
        "cohort": cohorts_da,
        "dt": dt_da,
    }
)
forcings

## 3. État Initial

**Variables d'état:**
- `Micronekton/biomass`: Biomasse totale du zooplancton (g/m²)
- `Micronekton/production`: Production structurée par cohorte (g/m²/s)

**Note:** Biomasse initiale à 0 - la croissance démarre via la production primaire

In [ ]:
# Biomasse initiale nulle
biomass_init = xr.DataArray(
    np.zeros((len(lats), len(lons))),
    coords={Coordinates.Y.value: lats, Coordinates.X.value: lons},
    dims=(Coordinates.Y.value, Coordinates.X.value),
    name="Micronekton/biomass",
)

# Production initiale nulle
production_init = xr.DataArray(
    np.zeros((len(lats), len(lons), len(cohorts))),
    coords={
        Coordinates.Y.value: lats,
        Coordinates.X.value: lons,
        "cohort": cohorts.magnitude,
    },
    dims=(Coordinates.Y.value, Coordinates.X.value, "cohort"),
    name="Micronekton/production",
)

# État initial : uniquement les variables d'état
# dt et cohort sont maintenant dans forcings
initial_state = xr.Dataset(
    {
        "Micronekton/biomass": biomass_init,
        "Micronekton/production": production_init,
    }
)
initial_state

## 4. Construction du Modèle (Blueprint)

**Architecture du modèle LMTL:**
1. **compute_day_length**: Calcule la longueur du jour (pour migration jour/nuit)
2. **compute_layer_weighted_mean**: Moyenne pondérée de la température sur les couches
3. **compute_threshold_temperature**: Applique un seuil minimal de température
4. **compute_gillooly_temperature**: Transformation de Gillooly pour les taux métaboliques
5. **compute_recruitment_age**: Âge de recrutement dépendant de la température
6. **compute_production_initialization**: Initialise la production à partir de la NPP
7. **compute_production_dynamics**: Évolution temporelle de la production par cohorte
8. **compute_mortality_tendency**: Mortalité dépendante de la température

In [ ]:
def configure_model(bp):
    # Forçages externes
    bp.register_forcing(
        "temperature",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value, Coordinates.Z.value),
        units="degree_Celsius",
    )
    bp.register_forcing(
        "primary_production",
        dims=(Coordinates.T.value, Coordinates.Y.value, Coordinates.X.value),
        units="g/m**2/second",
    )

    # Variables système
    bp.register_forcing("dt")
    bp.register_forcing("cohort")
    bp.register_forcing(Coordinates.T.value)
    bp.register_forcing(Coordinates.Y.value)

    bp.register_group(
        group_prefix="Micronekton",
        units=[
            {
                "func": compute_day_length,
                "output_mapping": {"output": "day_length"},
                "input_mapping": {"latitude": Coordinates.Y.value, "time": Coordinates.T.value},
                "output_units": {"output": "dimensionless"},
            },
            {
                "func": compute_layer_weighted_mean,
                "input_mapping": {"forcing": "temperature"},
                "output_mapping": {"output": "mean_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_threshold_temperature,
                "input_mapping": {"temperature": "mean_temperature", "min_temperature": "T_ref"},
                "output_mapping": {"output": "thresholded_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_gillooly_temperature,
                "input_mapping": {"temperature": "thresholded_temperature"},
                "output_mapping": {"output": "gillooly_temperature"},
                "output_units": {"output": "degree_Celsius"},
            },
            {
                "func": compute_recruitment_age,
                "input_mapping": {"temperature": "gillooly_temperature"},
                "output_mapping": {"output": "recruitment_age"},
                "output_units": {"output": "second"},
            },
            {
                "func": compute_production_initialization,
                "input_mapping": {"cohorts": "cohort"},
                "output_mapping": {"output": "production_source_npp"},
                "output_tendencies": {"output": "production"},
                "output_units": {"output": "g/m**2/second"},
            },
            {
                "func": compute_production_dynamics,
                "input_mapping": {
                    "cohort_ages": "cohort",
                    "dt": "dt",
                },
                "output_mapping": {
                    "production_tendency": "production_tendency",
                    "recruitment_source": "biomass_source",
                },
                "output_tendencies": {
                    "production_tendency": "production",
                    "recruitment_source": "biomass",
                },
                "output_units": {
                    "production_tendency": "g/m**2/second",
                    "recruitment_source": "g/m**2/second",
                },
            },
            {
                "func": compute_mortality_tendency,
                "input_mapping": {"temperature": "gillooly_temperature"},
                "output_mapping": {"mortality_loss": "biomass_mortality"},
                "output_tendencies": {"mortality_loss": "biomass"},
                "output_units": {"mortality_loss": "g/m**2/second"},
            },
        ],
        parameters={
            "day_layer": {"units": "dimensionless"},
            "night_layer": {"units": "dimensionless"},
            "tau_r_0": {"units": "second"},
            "gamma_tau_r": {"units": "1/degree_Celsius"},
            "lambda_0": {"units": "1/second"},
            "gamma_lambda": {"units": "1/degree_Celsius"},
            "T_ref": {"units": "degree_Celsius"},
            "E": {"units": "dimensionless"},
        },
        state_variables={
            "production": {
                "dims": (Coordinates.Y.value, Coordinates.X.value, "cohort"),
                "units": "g/m**2/second",
            },
            "biomass": {
                "dims": (Coordinates.Y.value, Coordinates.X.value),
                "units": "g/m**2",
            },
        },
    )

In [ ]:
Markdown("```mermaid\n" + bp.export_mermaid() + "\n```")

## 5. Exécution

**Configuration:**
- Backend: sequential (pas de parallélisation)
- Variables sauvegardées: biomass et production
- Intégration temporelle avec Euler forward

In [ ]:
# Initialisation du contrôleur
controller = SimulationController(config, backend="sequential")
controller.setup(
    configure_model,
    forcings,
    initial_state,
    parameters={"Micronekton": lmtl_params},
    output_variables=["Micronekton/biomass", "Micronekton/production"],  # Noms préfixés
)

# Exécution de la simulation
print(f"Starting simulation from {config.start_date} to {config.end_date}")
controller.run()

# Récupération des résultats (MemoryWriter par défaut)
full_history = controller.results
full_history

## 6. Visualisation

**Analyses:**
1. Évolution temporelle de la biomasse totale
2. Évolution de la production par cohorte
3. Distribution finale de la production par âge

In [ ]:
# Sélection d'un point spatial (milieu de la grille)
y_idx = len(full_history[Coordinates.Y.value]) // 2
x_idx = len(full_history[Coordinates.X.value]) // 2

# 1. Évolution de la Biomasse Totale au cours du temps
plt.figure(figsize=(12, 6))
total_biomass = full_history["Micronekton/biomass"].isel(
    {Coordinates.Y.value: y_idx, Coordinates.X.value: x_idx}
)
# Conversion g/m² -> kg/m² pour l'affichage
(total_biomass / 1000).plot(marker="o")
plt.title("Évolution de la Biomasse Totale (un point)")
plt.ylabel("Biomasse (kg/m²)")
plt.grid(True)
plt.tight_layout()
plt.show()

# 2. Évolution de la Production par Cohorte
plt.figure(figsize=(12, 6))
# On prend quelques cohortes représentatives
num_cohorts = len(full_history.cohort)
cohorts_to_plot = [0, num_cohorts // 2, num_cohorts - 1]

for c in cohorts_to_plot:
    prod_c = (
        full_history["Micronekton/production"]
        .isel(cohort=c)
        .isel(
            {
                Coordinates.Y.value: y_idx,
                Coordinates.X.value: x_idx,
            }
        )
        .sel({Coordinates.T.value: slice("2020-01-01", "2020-01-20")})
    )
    cohort_age_days = full_history.cohort.values[c] / 86400  # Conversion secondes -> jours
    prod_c.plot(label=f"Cohorte {c} ({cohort_age_days:.1f} jours)")

plt.title("Production par Cohorte au cours du temps")
plt.ylabel("Production (g/m²/s)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 3. Distribution de la Production par Cohorte (snapshot final)
plt.figure(figsize=(12, 6))
final_production = full_history["Micronekton/production"].isel(
    {Coordinates.T.value: -1, Coordinates.Y.value: y_idx, Coordinates.X.value: x_idx}
)
cohort_ages_days = full_history.cohort.values / 86400  # Conversion secondes -> jours
plt.bar(cohort_ages_days, final_production.values, width=0.8, alpha=0.7)
plt.xlabel("Âge de la cohorte (jours)")
plt.ylabel("Production (g/m²/s)")
plt.title("Distribution de la Production par Cohorte (état final)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()